# 臺北榮民總醫院 SmartCoder 正式 API

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dechnology/smartcoder-hospital-colab/blob/main/colab/hospitals/tvgh_smartcoder_api.ipynb)

- 院別代號：`tvgh`
- API base URL：`https://fhircsh.itri-nlp.tw/code_api/tvgh`
- 環境：正式服務
- 驗證狀態：正式入口已設定；完整病例整理流程待重新驗證。

執行第一個程式區塊時，請輸入該院 API key。Notebook 不會儲存或顯示 key。

In [ ]:
from getpass import getpass
from uuid import uuid4
import json
import os
import time

import requests

HOSPITAL_SLUG = "tvgh"
BASE_URL = "https://fhircsh.itri-nlp.tw/code_api/tvgh"
SERVICE_ENABLED = True
SERVICE_STATUS = "正式入口已設定；完整病例整理流程待重新驗證。"
COLAB_ORIGIN = "https://colab.research.google.com"

CODE_URL = f"{BASE_URL}/api/v1/snomed/coding"
RESULT_URL = f"{BASE_URL}/api/v1/snomed/results/{{request_id}}"

API_KEY = os.environ.get("SMARTCODER_API_KEY", "").strip()
if not API_KEY:
    API_KEY = os.environ.get("API_AUTH_TOKEN", "").strip()
if SERVICE_ENABLED and not API_KEY:
    API_KEY = getpass(f"請輸入 {HOSPITAL_SLUG} 的 X-API-Key：").strip()
if SERVICE_ENABLED and not API_KEY:
    raise ValueError("沒有輸入 API key。")

HEADERS = {
    "Content-Type": "application/json",
    "X-API-Key": API_KEY,
    "X-Hospital-Slug": HOSPITAL_SLUG,
    "Origin": COLAB_ORIGIN,
}

print("Hospital:", HOSPITAL_SLUG)
print("BASE_URL:", BASE_URL)
print("Status:", SERVICE_STATUS)
print("API key loaded:", bool(API_KEY))

In [ ]:
def assert_colab_cors(response):
    actual = response.headers.get("Access-Control-Allow-Origin")
    assert actual == COLAB_ORIGIN, (
        f"CORS 不符：預期 {COLAB_ORIGIN}，實際 {actual!r}"
    )


def safe_result(payload):
    metadata = payload.get("processing_metadata") or {}
    public_metadata = {
        key: metadata.get(key)
        for key in (
            "snomed_version",
            "pipeline_version",
            "vote_attempts",
            "confidence_method",
        )
    }
    return {
        "request_id": payload.get("request_id"),
        "polished_clinical_note": payload.get("polished_clinical_note"),
        "snomed_codings": payload.get("snomed_codings", []),
        "processing_metadata": public_metadata,
    }


def run_full_flow_case():
    request_id = str(uuid4())
    payload = {
        "request_id": request_id,
        "raw_clinical_note": "患者胸痛持續兩週，否認咳嗽。",
        "output_format": "simple",
    }

    started_at = time.perf_counter()
    post_response = requests.post(
        CODE_URL,
        headers=HEADERS,
        json=payload,
        timeout=240,
    )
    elapsed_seconds = time.perf_counter() - started_at
    print(
        f"POST HTTP {post_response.status_code}｜完整流程 "
        f"{elapsed_seconds:.2f} 秒"
    )
    post_response.raise_for_status()
    assert_colab_cors(post_response)

    post_payload = post_response.json()
    assert post_payload.get("request_id") == request_id, "POST request_id 不一致"
    assert (post_payload.get("polished_clinical_note") or "").strip(), "缺少整理後病歷"
    assert post_payload.get("snomed_codings"), "POST 編碼結果不得為空"

    metadata = post_payload.get("processing_metadata") or {}
    assert str(metadata.get("pipeline_version", "")).endswith("|txt_ner"), (
        "不是完整 TXT_NER 流程"
    )
    assert metadata.get("vote_attempts") == 3, "NER 必須執行 3 輪"
    assert all(
        item.get("source") == ["TXT_NER"]
        for item in post_payload["snomed_codings"]
    ), "結果含有非 TXT_NER 來源"
    assert any(
        item.get("concept_id") == "29857009"
        for item in post_payload["snomed_codings"]
    ), "未辨識出胸痛 SNOMED 概念"

    get_response = requests.get(
        RESULT_URL.format(request_id=request_id),
        headers=HEADERS,
        timeout=60,
    )
    print("GET HTTP", get_response.status_code)
    get_response.raise_for_status()
    assert_colab_cors(get_response)

    get_payload = get_response.json()
    assert get_payload.get("request_id") == request_id, "GET request_id 不一致"
    assert get_payload.get("status") == "completed", "GET 任務狀態不是 completed"
    assert (get_payload.get("response") or {}).get("snomed_codings") == (
        post_payload.get("snomed_codings")
    ), "GET 與 POST 編碼結果不一致"

    print(json.dumps(safe_result(post_payload), ensure_ascii=False, indent=2))

In [ ]:
if not SERVICE_ENABLED:
    raise RuntimeError(SERVICE_STATUS)

run_full_flow_case()
print("正式流程驗收通過")